In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import os
import sys
from sklearn.model_selection import train_test_split
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu" )
print(device)

cuda:3


In [2]:
class DF_XJTU():
    def __init__(self,args):
        self.normalization = True
        self.normalization_method = args.normalization_method # min-max, z-score
        self.args = args

    def _3_sigma(self, Ser1):
        rule = (Ser1.mean() - 3 * Ser1.std() > Ser1) | (Ser1.mean() + 3 * Ser1.std() < Ser1)
        index = np.arange(Ser1.shape[0])[rule]
        return index

    def delete_3_sigma(self,df):
        df = df.replace([np.inf, -np.inf], np.nan)
        df = df.dropna()
        df = df.reset_index(drop=True)
        out_index = []
        for col in df.columns:
            index = self._3_sigma(df[col])
            out_index.extend(index)
        out_index = list(set(out_index))
        df = df.drop(out_index, axis=0)
        df = df.reset_index(drop=True)
        return df

    def read_one_csv(self,file_name,nominal_capacity=None):
        df = pd.read_csv(file_name)
        df.insert(df.shape[1]-1,'cycle index',np.arange(df.shape[0]))

        df = self.delete_3_sigma(df)

        if nominal_capacity is not None:
            #print(f'nominal_capacity:{nominal_capacity}, capacity max:{df["capacity"].max()}',end=',')
            df['capacity'] = df['capacity']/nominal_capacity
            #print(f'SOH max:{df["capacity"].max()}')
            f_df = df.iloc[:,:-1]
            if self.normalization_method == 'min-max':
                f_df = 2*(f_df - f_df.min())/(f_df.max() - f_df.min()) - 1
            elif self.normalization_method == 'z-score':
                f_df = (f_df - f_df.mean())/f_df.std()

            df.iloc[:,:-1] = f_df

        return df

    def load_one_battery(self,path,nominal_capacity=None):
        df = self.read_one_csv(path,nominal_capacity)
        # XJTU数据特征选择
        # Var2：'CV charge time','cycle index'
        # Var3：'CV charge time','current entropy','cycle index'
        # Var17：'voltage mean','voltage std','voltage kurtosis','voltage skewness','CC Q','CC charge time','voltage slope','voltage entropy',
        # 'current mean','current std','current kurtosis','current skewness','CV Q','CV charge time','current slope','current entropy','cycle index','capacity'
        df = df.filter(items=['voltage mean','voltage std','voltage kurtosis','voltage skewness','CC Q','CC charge time','voltage slope','voltage entropy','current mean','current std','current kurtosis','current skewness','CV Q','CV charge time','current slope','current entropy','cycle index','capacity']) # CC Q在恒流充电时等价于CC charge time
        x = df.iloc[:,:-1].values
        y = df.iloc[:,-1].values
        x1 = x[:-1]
        x2 = x[1:]
        y1 = y[:-1]
        y2 = y[1:]
        return (x,y),(x1,y1),(x2,y2)

    def load_all_battery(self,path_list,nominal_capacity):
        X, Y, X1, X2, Y1, Y2 = [], [], [], [], [], []
        for path in path_list:
            (x, y),(x1, y1), (x2, y2) = self.load_one_battery(path, nominal_capacity)
            X.append(x)
            X1.append(x1)
            X2.append(x2)
            Y.append(y)
            Y1.append(y1)
            Y2.append(y2)

        X = np.concatenate(X, axis=0)
        X1 = np.concatenate(X1, axis=0)
        X2 = np.concatenate(X2, axis=0)
        Y = np.concatenate(Y, axis=0)
        Y1 = np.concatenate(Y1, axis=0)
        Y2 = np.concatenate(Y2, axis=0)

        tensor_X = torch.from_numpy(X).float().to(device) 
        tensor_X1 = torch.from_numpy(X1).float().to(device)
        tensor_X2 = torch.from_numpy(X2).float().to(device)
        tensor_Y = torch.from_numpy(Y).float().view(-1,1).to(device)
        tensor_Y1 = torch.from_numpy(Y1).float().view(-1,1).to(device)
        tensor_Y2 = torch.from_numpy(Y2).float().view(-1,1).to(device)

        train_X1, valid_X1, train_X2, valid_X2, train_Y1, valid_Y1, train_Y2, valid_Y2 = \
            train_test_split(tensor_X1, tensor_X2, tensor_Y1, tensor_Y2, test_size=0.2, random_state=420)
        train_loader = DataLoader(TensorDataset(train_X1, train_X2, train_Y1, train_Y2),
                                  batch_size=self.args.batch_size,
                                  shuffle=True)
        valid_loader = DataLoader(TensorDataset(valid_X1, valid_X2, valid_Y1, valid_Y2),
                                  batch_size=self.args.batch_size,
                                  shuffle=True)
        test_loader = DataLoader(TensorDataset(tensor_X1, tensor_X2, tensor_Y1, tensor_Y2),
                                 batch_size=self.args.batch_size,
                                 shuffle=False)

        data = {'input': tensor_X, 'label': tensor_Y,
                  'train_loader': train_loader,
                  'valid_loader': valid_loader,
                  'test_loader': test_loader}

        return data

In [3]:
class XJTUdataFilter(DF_XJTU):
    def __init__(self,root='../data/XJTU data',args=None):
        super(XJTUdataFilter, self).__init__(args)
        self.root = root
        self.file_list = os.listdir(root)
        self.variables = pd.read_csv(os.path.join(root, self.file_list[0])).columns
        self.num = len(self.file_list)
        self.batch_names = ['2C','3C','R2.5','R3','RW','satellite']
        self.batch_size = args.batch_size

        if self.normalization:
            self.nominal_capacity = 2.0
        else:
            self.nominal_capacity = None

    def read_one_batch(self,batch='2C'):
        if isinstance(batch,int):
            batch = self.batch_names[batch]
        assert batch in self.batch_names, 'batch must be in {}'.format(self.batch_names)
        file_list = []
        for i in range(self.num):
            if batch in self.file_list[i]:
                path = os.path.join(self.root,self.file_list[i])
                file_list.append(path)
        return self.load_all_battery(path_list=file_list,nominal_capacity=self.nominal_capacity)

    def read_all(self,specific_path_list=None):
        if specific_path_list is None:
            file_list = []
            for file in self.file_list:
                path = os.path.join(self.root, file)
                file_list.append(path)
            return self.load_all_battery(path_list=file_list,nominal_capacity=self.nominal_capacity)
        else:
            return self.load_all_battery(path_list=specific_path_list,nominal_capacity=self.nominal_capacity)

In [4]:
def load_XJTU_data_filter(args,small_sample=None):   # 无差分，差分：diff
    root = '../data/XJTU data'
    data = XJTUdataFilter(root=root, args=args)
    train_list = []
    test_list = []
    files = os.listdir(root)
    for file in files:
        if args.batch in file:
            if '4' in file or '8' in file:
                test_list.append(os.path.join(root, file))
            else:
                train_list.append(os.path.join(root, file))
    if small_sample is not None:
        train_list = train_list[:small_sample]

    train_data = data.read_all(specific_path_list=train_list)
    test_data = data.read_all(specific_path_list=test_list)
    
    dataset = {'train_input':train_data['input'],
                  'train_label':train_data['label'],
                  'test_input':test_data['input'],
                  'test_label':test_data['label'],
                  'train_loader':train_data['train_loader'],
                  'valid_loader':train_data['valid_loader'],
                  'test_loader':test_data['test_loader']}
    return dataset

In [5]:
import argparse
parser = argparse.ArgumentParser('存储过滤出关键健康指标的HUST数据集')
parser.add_argument('--dataset',type=str,default='XJTU',choices=['XJTU','HUST','MIT','TJU'])
parser.add_argument('--data_root', type=str, default='../data/XJTU data', help='HUST数据集根路径')
parser.add_argument('--batch',type=str,default='2C',choices=['2C','3C','R2.5','R3','RW','satellite'])
parser.add_argument('--normalization_method',type=str, default='min-max', help='min-max,z-score')
parser.add_argument('--batch_size',type=int,default=256)
args, _ = parser.parse_known_args()

In [6]:
dataset = load_XJTU_data_filter(args)
torch.save(dataset, '../Data/XJTU_Data_Var17_batch_0.pt')

/tmp/ipykernel_2915674/4270025699.py:41: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0     -1.000000
1     -0.994872
2     -0.989744
3     -0.984615
4     -0.979487
         ...   
366    0.979487
367    0.984615
368    0.989744
369    0.994872
370    1.000000
Name: cycle index, Length: 371, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.iloc[:,:-1] = f_df
/tmp/ipykernel_2915674/4270025699.py:41: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0     -1.000000
1     -0.994695
2     -0.989390
3     -0.984085
4     -0.978780
         ...   
354    0.978780
355    0.984085
356    0.989390
357    0.994695
358    1.000000
Name: cycle index, Length: 359, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.iloc[:,:-1] = f_df
/tmp/ipykern

In [7]:
import gc
torch.cuda.empty_cache()
gc.collect()

312